In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
spark.sql("use catalog datatocrunch_novacart_adb ")

In [0]:
spark.sql("create schema if not exists bronze_schema")

In [0]:
spark.sql("""
    create table if not exists datatocrunch_novacart_adb.bronze_schema.ingestion_control(
        layer string,
        table_name string,
        ts_col string,
        pk_col string,
        last_successful_ts timestamp,
        last_successful_pk bigint,
        last_run_id string,
        rows_written bigint,
        run_status string,
        updated_at timestamp
    )
    using delta
""")


In [0]:
tables_config = {
    "orders"   : {"pk_col" : "order_id"  , "ts_col" : "updated_at"},
    "products" : {"pk_col" : "product_id", "ts_col" : "updated_at"},
    "payments" : {"pk_col" : "payment_id", "ts_col" : "processed_at"}
}

bronze_run_id = str(uuid.uuid4())
print("Current Bronze Run ID:", bronze_run_id)


In [0]:
def get_last_successful_watermark(table_name: str):
    ctrl = (
        spark.table("datatocrunch_novacart_adb.bronze_schema.ingestion_control")
        .filter(
            (f.col("layer") == "bronze") &
            (f.col("table_name") == table_name) &
            (f.col("run_status") == "success")
        )
        .orderBy(f.col("updated_at").desc())
        .limit(1)
    )
    
    rows = ctrl.collect()
    if not rows:
        return None, None
        
    return rows[0]["last_successful_ts"], rows[0]["last_successful_pk"]


In [0]:
from datetime import datetime
from delta.tables import DeltaTable

def upsert_bronze_control(
    table_name,
    ts_col,
    pk_col,
    last_ts,
    last_pk,
    rows_written,
    run_id
):
    # Create the single-row control DataFrame
    control_df = spark.createDataFrame(
        [
            (
                "bronze",
                table_name,
                ts_col,
                pk_col,
                last_ts,
                int(last_pk) if last_pk is not None else None,
                run_id,
                int(rows_written),
                "success",
                datetime.now()
            )
        ],
        schema="""
            layer string,
            table_name string,
            ts_col string,
            pk_col string,
            last_successful_ts timestamp,
            last_successful_pk bigint,
            last_run_id string,
            rows_written bigint,
            run_status string,
            updated_at timestamp
        """
    )

    # Target Delta table
    dt = DeltaTable.forName(
        spark,
        "datatocrunch_novacart_adb.bronze_schema.ingestion_control"
    )

    # MERGE / UPSERT
    (
        dt.alias("t")
        .merge(
            control_df.alias("s"),
            "t.table_name = s.table_name AND t.layer = s.layer"
        )
        .whenMatchedUpdate(
            set={
                "ts_col": "s.ts_col",
                "pk_col": "s.pk_col",
                "last_successful_ts": "s.last_successful_ts",
                "last_successful_pk": "s.last_successful_pk",
                "last_run_id": "s.last_run_id",
                "rows_written": "s.rows_written",
                "run_status": "s.run_status",
                "updated_at": "s.updated_at"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
import pyspark.sql.functions as f


In [0]:
spark.sql("SHOW CATALOGS").show(truncate=False)


In [0]:
%sql
CREATE TABLE IF NOT EXISTS datatocrunch_novacart_adb.bronze_schema.ingestion_control (
    table_name STRING,
    ts_col STRING,
    pk_col STRING,
    last_successful_ts TIMESTAMP,
    last_successful_pk STRING,
    row_count LONG,
    bronze_run_id STRING
)
USING DELTA;


In [0]:
# Run this right before your loop to force the catalog/schema scope
spark.sql("USE CATALOG datatocrunch_novacart_adb")
spark.sql("USE SCHEMA bronze_schema")

# Your Loop Code
for table_name, cfg in tables_config.items():
    pk_col = cfg["pk_col"]
    ts_col = cfg["ts_col"]
    
    source_table = f"datatocruch_novacart_sql_connection_catalog.dbo.{table_name}"
    target_table = f"datatocrunch_novacart_adb.bronze_schema.{table_name}_raw"
    
    last_successful_ts, last_successful_pk = get_last_successful_watermark(table_name)
    # ... rest of your code ...

    print(f"\n=== Processing {table_name} ===")
    print(f"last_successful_ts = {last_successful_ts}")
    print(f"last_successful_pk = {last_successful_pk}")
    
    source_df = spark.read.table(source_table).withColumn(ts_col, F.col(ts_col).cast("timestamp"))
    
    if last_successful_ts is None:
        rows_to_load = source_df
    else:
        rows_to_load = source_df.filter(
            (F.col(ts_col) > F.lit(last_successful_ts)) |
            (
                (F.col(ts_col) == F.lit(last_successful_ts)) &
                (F.col(pk_col).cast("long") > F.lit(int(last_successful_pk)))
            )
        )
        
    rows_to_load = (
        rows_to_load
        .withColumn("bronze_ingested_at", F.current_timestamp())
        .withColumn("bronze_run_id", F.lit(bronze_run_id))
        .withColumn("bronze_source_table", F.lit(source_table))
    )
    
    row_count = rows_to_load.count()
    print(f"{table_name} rows_to_load = {row_count}")
    
    if row_count == 0:
        print(f"No new rows for {table_name}.")
        upsert_bronze_control(
            table_name,
            ts_col,
            pk_col,
            last_successful_ts,
            last_successful_pk,
            row_count,
            bronze_run_id
        )
        continue

    rows_to_load.write.format("delta").mode("append").saveAsTable(target_table)
    
    max_ts = rows_to_load.agg(F.max(ts_col).alias("max_ts")).first()["max_ts"]
    
    max_pk = (
        rows_to_load
        .filter(F.col(ts_col) == F.lit(max_ts))
        .agg(F.max(F.col(pk_col).cast("long")).alias("max_pk"))
        .first()["max_pk"]
    )
    
    upsert_bronze_control(table_name, ts_col, pk_col, max_ts, max_pk, row_count, bronze_run_id)
    print(f"Wrote {row_count} rows to {target_table}")


In [0]:
display(spark.sql("SHOW SCHEMAS IN datatocrunch_novacart_adb"))


In [0]:
# Fetches metrics safely without memory bloat
orders_count = spark.sql("SELECT count(*) FROM datatocrunch_novacart_adb.bronze_schema.orders_raw").first()[0]
products_count = spark.sql("SELECT count(*) FROM datatocrunch_novacart_adb.bronze_schema.products_raw").first()[0]
payments_count = spark.sql("SELECT count(*) FROM datatocrunch_novacart_adb.bronze_schema.payments_raw").first()[0]

print(f"Orders Bronze Count   : {orders_count}")
print(f"Products Bronze Count : {products_count}")
print(f"Payments Bronze Count : {payments_count}")

display(spark.sql("SELECT * FROM datatocrunch_novacart_adb.bronze_schema.ingestion_control").orderBy("table_name"))
